# YOLOv8 Kurgan Detection: Colab Training From GitHub + Google Drive

This notebook is designed for Google Colab.

Workflow:

1. Mount Google Drive.
2. Clone the GitHub repository.
3. Load the YOLO bbox dataset from Google Drive.
4. Optionally build filtered dataset variants `v2` or `v4`.
5. Train YOLOv8.
6. Run optional threshold validation.
7. Save training artifacts as a zip archive back to Google Drive.

The full dataset, model weights, and `runs/` directories should stay outside GitHub.

## 1. Configuration

Edit this cell before running. The default values reproduce the current `v4` balanced experiment.

In [ ]:
from pathlib import Path

# GitHub repository
REPO_URL = "https://github.com/MataNerdy/Geodata_Archaeology_CV.git"
BRANCH = "main"
PROJECT_SUBDIR = "04_detection_yolo"

# Dataset on Google Drive.
# Expected zip content can be either:
#   dataset_yolo_bbox/
# or a ready filtered dataset folder with images/, labels/, dataset.yaml.
DRIVE_DATASET_ZIP = "/content/drive/MyDrive/Share/Geodata/dataset_yolo_bbox.zip"
DATA_ROOT = Path("/content/dataset")
SOURCE_DATASET_DIR = DATA_ROOT / "dataset_yolo_bbox"

# Choose one:
#   "v2"       -> kurgan-only Li/Ae baseline
#   "v4"       -> balanced clean Li/Ae dataset
#   "prebuilt" -> use SOURCE_DATASET_DIR as-is
DATASET_VARIANT = "v4"

# YOLO training config
MODEL_NAME = "yolov8s.pt"
IMGSZ = 1024
EPOCHS = 80
BATCH = 8
PATIENCE = 25
CLOSE_MOSAIC = 15
WORKERS = 2
CACHE = True
COS_LR = True
RUN_NAME = "kurgans_li_ae_v4_yolov8s_balanced_colab"

# Optional validation at several confidence thresholds.
RUN_THRESHOLD_VALIDATION = True
CONF_THRESHOLDS = [0.10, 0.15, 0.25]

# Archive config
DRIVE_OUTPUT_DIR = Path("/content/drive/MyDrive/Share/Geodata/yolo_detection_runs")
ARCHIVE_PREFIX = "yolo_kurgan_detection"
INCLUDE_WEIGHTS_IN_ARCHIVE = True

print("Configured dataset variant:", DATASET_VARIANT)
print("Drive dataset zip:", DRIVE_DATASET_ZIP)
print("Output dir:", DRIVE_OUTPUT_DIR)

## 2. Mount Google Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 3. Clone GitHub Repository And Install Dependencies

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

REPO_DIR = Path("/content/Geodata_Archaeology_CV")

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)

PROJECT_DIR = REPO_DIR / PROJECT_SUBDIR
os.chdir(PROJECT_DIR)
print("Project dir:", PROJECT_DIR)

subprocess.run(["python", "-m", "pip", "install", "-q", "--upgrade", "pip"], check=True)
subprocess.run(["python", "-m", "pip", "install", "-q", "ultralytics", "pandas", "numpy", "pyyaml", "pillow"], check=True)

print("Repository ready")

## 4. Load Dataset From Google Drive

In [ ]:
import zipfile
from pathlib import Path

DATA_ROOT.mkdir(parents=True, exist_ok=True)

zip_path = Path(DRIVE_DATASET_ZIP)
if not zip_path.exists():
    raise FileNotFoundError(f"Dataset zip not found: {zip_path}")

print("Unzipping:", zip_path)
with zipfile.ZipFile(zip_path, "r") as zf:
    zf.extractall(DATA_ROOT)

print("Dataset root content:")
for p in sorted(DATA_ROOT.iterdir()):
    print(" -", p)

if not SOURCE_DATASET_DIR.exists():
    candidates = [p for p in DATA_ROOT.iterdir() if p.is_dir() and (p / "dataset.yaml").exists()]
    if len(candidates) == 1:
        SOURCE_DATASET_DIR = candidates[0]
        print("Auto-detected dataset dir:", SOURCE_DATASET_DIR)
    else:
        raise FileNotFoundError(f"Could not find source dataset dir: {SOURCE_DATASET_DIR}")

print("Source dataset:", SOURCE_DATASET_DIR)

## 5. Build Filtered Dataset Variant

This cell materializes `v2` or `v4` from the original `dataset_yolo_bbox`. If `DATASET_VARIANT = "prebuilt"`, it uses the dataset from Drive unchanged.

In [ ]:
from pathlib import Path
import random
import shutil
import pandas as pd

KEEP_MODALITIES = {"Li", "Ae"}
KEEP_OLD_CLASS_IDS = {0, 1}
CLASS_ID_MAP = {0: 0, 1: 1}
NAMES = {0: "kurgany_tselye", 1: "kurgany_povrezhdennye"}

def make_dirs(out_dir: Path):
    for split in ["train", "val"]:
        (out_dir / "images" / split).mkdir(parents=True, exist_ok=True)
        (out_dir / "labels" / split).mkdir(parents=True, exist_ok=True)

def resolve_old_path(path_from_meta, old_dir, kind, split):
    p = Path(str(path_from_meta))
    if p.exists():
        return p
    candidate = old_dir / kind / split / p.name
    if candidate.exists():
        return candidate
    return p

def parse_yolo_label(path: Path):
    boxes = []
    if not path.exists():
        return boxes
    for line in path.read_text(encoding="utf-8").splitlines():
        parts = line.strip().split()
        if len(parts) != 5:
            continue
        cls_id = int(float(parts[0]))
        coords = list(map(float, parts[1:]))
        if cls_id not in KEEP_OLD_CLASS_IDS:
            continue
        boxes.append((CLASS_ID_MAP[cls_id], *coords))
    return boxes

def valid_box(box):
    _, xc, yc, w, h = box
    return 0 <= xc <= 1 and 0 <= yc <= 1 and 0 < w <= 1 and 0 < h <= 1

def write_label(path: Path, boxes):
    lines = [f"{cls_id} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}" for cls_id, xc, yc, w, h in boxes]
    path.write_text("\n".join(lines), encoding="utf-8")

def write_yaml(out_dir: Path):
    text = f"""path: {out_dir.resolve()}

train: images/train
val: images/val

names:
  0: kurgany_tselye
  1: kurgany_povrezhdennye
"""
    (out_dir / "dataset.yaml").write_text(text, encoding="utf-8")

def build_filtered_dataset(old_dir: Path, variant: str):
    if variant == "v2":
        out_dir = DATA_ROOT / "dataset_yolo_bbox_v2_kurgans_li_ae"
        negative_ratio = 0.25
        min_valid_fraction = None
        min_contrast = None
        max_objects = None
        edge_ratio_limit = None
    elif variant == "v4":
        out_dir = DATA_ROOT / "dataset_yolo_bbox_v4_kurgans_li_ae_balanced"
        negative_ratio = 0.15
        min_valid_fraction = 0.25
        min_contrast = 3
        max_objects = 20
        edge_ratio_limit = 0.8
    elif variant == "prebuilt":
        return old_dir
    else:
        raise ValueError(f"Unknown DATASET_VARIANT: {variant}")

    if out_dir.exists():
        shutil.rmtree(out_dir)

    random.seed(42)
    make_dirs(out_dir)

    meta_path = old_dir / "metadata.csv"
    if not meta_path.exists():
        raise FileNotFoundError(f"metadata.csv is required to build {variant}: {meta_path}")

    meta = pd.read_csv(meta_path)
    images = meta.drop_duplicates("image").copy()
    images = images[images["modality"].isin(KEEP_MODALITIES)].copy()

    new_rows = []
    copied_images = 0
    positive_images = 0
    negative_images = 0
    total_boxes = 0
    bad_boxes = 0
    skipped_missing = 0

    for _, row in images.iterrows():
        split = row["split"]
        old_img = resolve_old_path(row["image"], old_dir, "images", split)
        old_lbl = resolve_old_path(row["label"], old_dir, "labels", split)

        if not old_img.exists():
            skipped_missing += 1
            continue

        if edge_ratio_limit is not None and "bbox_touches_tile_edge" in meta.columns:
            same_image = meta[meta["image"] == row["image"]]
            edge_ratio = same_image["bbox_touches_tile_edge"].fillna(False).mean()
            if edge_ratio > edge_ratio_limit:
                continue

        if min_valid_fraction is not None and "valid_fraction" in row and row["valid_fraction"] < min_valid_fraction:
            continue

        if min_contrast is not None and "tile_p98_minus_p2" in row and row["tile_p98_minus_p2"] < min_contrast:
            continue

        boxes = parse_yolo_label(old_lbl)
        boxes_ok = []
        for box in boxes:
            if valid_box(box):
                boxes_ok.append(box)
            else:
                bad_boxes += 1

        if max_objects is not None and len(boxes_ok) > max_objects:
            continue

        is_positive = len(boxes_ok) > 0
        if not is_positive and random.random() > negative_ratio:
            continue

        new_img = out_dir / "images" / split / old_img.name
        new_lbl = out_dir / "labels" / split / old_lbl.name
        shutil.copy2(old_img, new_img)
        write_label(new_lbl, boxes_ok)

        copied_images += 1
        total_boxes += len(boxes_ok)
        positive_images += int(is_positive)
        negative_images += int(not is_positive)

        base = row.to_dict()
        base["image"] = str(new_img)
        base["label"] = str(new_lbl)
        base["is_positive"] = bool(is_positive)
        base["n_objects"] = len(boxes_ok)

        if boxes_ok:
            for cls_id, xc, yc, w, h in boxes_ok:
                obj = base.copy()
                obj.update({"class_id": cls_id, "class_name": NAMES[cls_id], "yolo_xc": xc, "yolo_yc": yc, "yolo_w": w, "yolo_h": h})
                new_rows.append(obj)
        else:
            obj = base.copy()
            obj.update({"class_id": None, "class_name": None, "yolo_xc": None, "yolo_yc": None, "yolo_w": None, "yolo_h": None})
            new_rows.append(obj)

    new_meta = pd.DataFrame(new_rows)
    new_meta.to_csv(out_dir / "metadata.csv", index=False)
    write_yaml(out_dir)

    print("=" * 80)
    print(f"DONE: {variant}")
    print("dataset:", out_dir)
    print("images copied:", copied_images)
    print("positive images:", positive_images)
    print("negative images:", negative_images)
    print("total boxes:", total_boxes)
    print("bad boxes skipped:", bad_boxes)
    print("missing images skipped:", skipped_missing)
    if not new_meta.empty:
        print("\nBBoxes by class:")
        print(new_meta[new_meta["is_positive"]].groupby("class_name").size())

    return out_dir

TRAIN_DATASET_DIR = build_filtered_dataset(SOURCE_DATASET_DIR, DATASET_VARIANT)
DATA_YAML = TRAIN_DATASET_DIR / "dataset.yaml"

print("Training dataset:", TRAIN_DATASET_DIR)
print(DATA_YAML.read_text(encoding="utf-8"))

## 6. Train YOLOv8

In [ ]:
from ultralytics import YOLO

model = YOLO(MODEL_NAME)

train_result = model.train(
    data=str(DATA_YAML),
    imgsz=IMGSZ,
    epochs=EPOCHS,
    batch=BATCH,
    patience=PATIENCE,
    cos_lr=COS_LR,
    workers=WORKERS,
    cache=CACHE,
    close_mosaic=CLOSE_MOSAIC,
    project="/content/runs/detect",
    name=RUN_NAME,
    exist_ok=True,
)

RUN_DIR = Path("/content/runs/detect") / RUN_NAME
BEST_WEIGHTS = RUN_DIR / "weights" / "best.pt"
if not BEST_WEIGHTS.exists():
    # Older Ultralytics versions sometimes save weights directly in run dir.
    alt = RUN_DIR / "best.pt"
    if alt.exists():
        BEST_WEIGHTS = alt

print("Run dir:", RUN_DIR)
print("Best weights:", BEST_WEIGHTS)

## 7. Optional Threshold Validation

In [ ]:
from ultralytics import YOLO

THRESHOLD_RUN_DIRS = []

if RUN_THRESHOLD_VALIDATION:
    if not BEST_WEIGHTS.exists():
        raise FileNotFoundError(f"Best weights not found: {BEST_WEIGHTS}")

    val_model = YOLO(str(BEST_WEIGHTS))
    for conf in CONF_THRESHOLDS:
        val_name = f"{RUN_NAME}_val_conf_{str(conf).replace('.', '_')}"
        metrics = val_model.val(
            data=str(DATA_YAML),
            imgsz=IMGSZ,
            conf=conf,
            project="/content/runs/detect",
            name=val_name,
            exist_ok=True,
            plots=True,
        )
        out_dir = Path("/content/runs/detect") / val_name
        THRESHOLD_RUN_DIRS.append(out_dir)
        print("conf:", conf, "save_dir:", out_dir)
else:
    print("Threshold validation skipped")

## 8. Summarize Training Metrics

In [ ]:
import pandas as pd

results_csv = RUN_DIR / "results.csv"
if results_csv.exists():
    df = pd.read_csv(results_csv)
    df.columns = [c.strip() for c in df.columns]
    metric_cols = ["epoch", "metrics/precision(B)", "metrics/recall(B)", "metrics/mAP50(B)", "metrics/mAP50-95(B)"]
    metric_cols = [c for c in metric_cols if c in df.columns]
    print("Last epoch metrics:")
    display(df[metric_cols].tail(1))
    if "metrics/mAP50(B)" in df.columns:
        best_idx = df["metrics/mAP50(B)"].idxmax()
        print("Best mAP50 epoch metrics:")
        display(df.loc[[best_idx], metric_cols])
else:
    print("results.csv not found:", results_csv)

## 9. Archive Results To Google Drive

The archive includes training plots, CSV/YAML files, validation images, and optionally model weights.

In [ ]:
from datetime import datetime
import zipfile

DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
archive_path = DRIVE_OUTPUT_DIR / f"{ARCHIVE_PREFIX}_{DATASET_VARIANT}_{RUN_NAME}_{timestamp}.zip"

include_suffixes = {".csv", ".yaml", ".yml", ".png", ".jpg", ".jpeg", ".txt"}
if INCLUDE_WEIGHTS_IN_ARCHIVE:
    include_suffixes.add(".pt")

dirs_to_archive = [RUN_DIR] + THRESHOLD_RUN_DIRS

with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for directory in dirs_to_archive:
        if not directory.exists():
            continue
        for file in directory.rglob("*"):
            if not file.is_file():
                continue
            if file.suffix.lower() not in include_suffixes:
                continue
            arcname = file.relative_to(Path("/content/runs/detect"))
            zf.write(file, arcname=arcname)

    # Save dataset metadata/yaml used for this training run.
    for file in [DATA_YAML, TRAIN_DATASET_DIR / "metadata.csv"]:
        if file.exists():
            zf.write(file, arcname=Path("dataset") / file.name)

print("Archive saved:", archive_path)
print("Archive size MB:", round(archive_path.stat().st_size / (1024 * 1024), 2))

## 10. Download Archive Manually If Needed

The archive is already saved to Google Drive. This optional cell also exposes it for direct browser download from Colab.

In [ ]:
from google.colab import files

# Uncomment if you want Colab to download the archive directly to your local machine.
# files.download(str(archive_path))

print("Drive archive:", archive_path)